# Notebook 05. ESG feature engineering and measurement validation

Notebook này thực hiện quá trình xây dựng các đặc trưng ESG từ dữ liệu Amazon dựa trên Feature Specification đã được xây dựng ở Notebook 04.

Mỗi ESG Indicator sẽ được chuyển đổi thành một hoặc nhiều đặc trưng định lượng thông qua các phương pháp Feature Engineering phù hợp như Dictionary Matching, Keyword Matching, Complaint Matching, Review Evidence Matching và Rule-based Matching.

Sau khi hoàn thành Feature Engineering, notebook tiến hành đánh giá khả năng đo lường thực nghiệm của từng ESG Indicator trên dữ liệu Amazon nhằm xác định tập indicator cuối cùng sử dụng cho phương pháp AHP.

Đầu vào

- cleaned_amazon_dataset.csv
- feature_specification.csv
- sustainable_material_dictionary.csv
- eco_label_dictionary.csv
- esg_keyword_dictionary.csv
- fashion_esg_complaint_ontology.csv

Đầu ra

- product_esg_feature_dataset.csv
- measurement_validation.csv
- final_indicator_for_ahp.csv

# Phần 0. Setup

Chuẩn bị môi trường làm việc cho notebook.

Phần này thực hiện:

- Import thư viện.
- Thiết lập hiển thị.
- Thiết lập tham số.
- Thiết lập đường dẫn dữ liệu đầu vào.
- Thiết lập đường dẫn dữ liệu đầu ra.

In [109]:
# Import libraries

import re

from pathlib import Path

from collections import Counter

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import nltk

from nltk.util import ngrams

from tqdm.auto import tqdm

In [110]:
# Display settings

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_colwidth",
    None
)

tqdm.pandas()

In [111]:
# Download NLTK resources

nltk.download(
    "punkt"
)

[nltk_data] Downloading package punkt to C:\Users\ASPIRE
[nltk_data]     7\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [112]:
# Project path

project_path = Path("..")

In [113]:
# Dataset paths

data_path = (
    project_path
    / "Data ESG"
)

amazon_review_path = (
    data_path
    / "amazon-fashion-800k+-user-reviews-dataset.csv"
)

fast_fashion_material_path = (
    data_path
    / "fastFashionCompDim.csv"
)

fast_fashion_item_path = (
    data_path
    / "fastFasionItemsDim.csv"
)

In [114]:
# ESG Knowledge Base paths

knowledge_base_path = (
    project_path
    / "Dataset"
    / "processed"
    / "ESG_Knowledge_Base"
)

esg_keyword_dictionary_path = (
    knowledge_base_path
    / "esg_keyword_dictionary.csv"
)

sustainable_material_dictionary_path = (
    knowledge_base_path
    / "sustainable_material_dictionary.csv"
)

eco_label_dictionary_path = (
    knowledge_base_path
    / "eco_label_dictionary.csv"
)

fashion_esg_complaint_ontology_path = (
    knowledge_base_path
    / "fashion_esg_complaint_ontology.csv"
)

In [115]:
# Feature Specification path

feature_specification_path = (
    project_path
    / "Dataset"
    / "processed"
    / "ESG_Operationalization"
    / "feature_specification.csv"
)

In [116]:
# Output path

seller_esg_feature_dataset_path = (
    project_path
    / "Dataset"
    / "processed"
    / "seller_esg_feature_dataset.csv"
)

In [117]:
# Setup completed

print("Setup completed successfully.")

Setup completed successfully.


# PHẦN 1. LOAD DATASETS

Nạp Seller-level Amazon Dataset đã được tiền xử lý để xây dựng các đặc trưng ESG.

Dataset này chứa thông tin về:

- Thông tin sản phẩm
- Thông tin người bán
- Thông tin đánh giá
- Nội dung văn bản sau khi làm sạch

Đây là nguồn dữ liệu chính để xây dựng Seller ESG Feature Dataset.

In [118]:
# Load cleaned Amazon Dataset

amazon_df = pd.read_csv(
    project_path
    / "Dataset"
    / "processed"
    / "cleaned_amazon_dataset.csv"
)

print(
    f"Dataset shape: {amazon_df.shape}"
)

display(
    amazon_df.head()
)

Dataset shape: (6327, 31)


,asin,about_item,product_description,availability,brand_name,manufacturer,price_value,rating_count,rating_stars,recent_purchases,seller_name,seller_page_url,rank_1,best_sellers_rank,productasin,productvariant,rating,reviewid,reviewmetadata,reviewtext,reviewtitle,verifiedpurchase,cleaned_review_text,sentiment_score,product_text,review_text_clean,has_review,has_product_info,review_length,product_text_length,is_verified_purchase
0,B0DLGB4RYH,"material: men's polo shirt is made of soft polyester fabric, moisture wicking, stain-resistant, durable work shirts, lightweight, breathable and comfortable. golf shirt for men keeps you cool and dry in all day.design: mens short sleeve polo shirts designed with 3 button closure, solid color basic collared t-shirt with split hem, unrestricted movement during your golf swings, fashion classic design golf polo shirt is great for spring, summer and fall.match: polo shirts can be worn out or tucked. tucked polos for business casual settings, while untucked polos casual look ideal for everyday wear. basic polo t shirts for men easy to match with slacks, jeans, suit pants, shorts, etc.occasions: polo shirts for men is suitable for casual, golf, business, office, work, everyday wear, school uniform, travel, meeting, dating, dinner, street, hiking, camping, tennis or other activities. perfect valentine's day, father's day, christmas, thanksgiving day, birthday gift for father, husband, son, boyfriends, friends or yourself.garment care: machine washable. easy to care and wash. note: men's golf polo shirt is in standard us size, please choose your size refer to the size information under the description before ordering.",NaN,In Stock,COOFANDY Store,unknown,19.9920,0.0000,0.0000,0.0000,COOFANDY,https://www.amazon.com/gp/help/seller/at-a-glance.html/ref=dp_merchant_link?ie=UTF8&seller=AW10J8VB8Z74G&asin=B0DLGB4RYH&ref_=dp_merchant_link&isAmazonFulfilled=1,199.0000,"#50,261 in Clothing, Shoes & Jewelry (See Top 100 in Clothing, Shoes & Jewelry) #199 in Men's Polo Shirts",B0DLGB4RYH,Color: BlackSize: X-Large,5.0000,R2AUQFPJY5ERCZ,"Reviewed in the United States on March 6, 2025","‚úçô∏è the coofandy men's polo shirt is a fantastic blend of style, comfort, and affordability, making it a great choice for casual or smart-casual occasions. made from 100% polyester, the shirt is lightweight, breathable, and moisture-wicking, ideal for warmer weather or active wear. the fit is slim but not clingy, with shorter sleeves that hug the upper arms, giving it a tailored and modern look. the collar is well-constructed and adds a polished touch without being overbearing.the shirt‚äôs material is cool to the touch and resists wrinkling, making it easy to maintain. the color options, like light blue, dark gray, and wine red, are vibrant and true to the product photos. however, the fabric is on the thinner side, so it may not be suitable for colder weather or those who prefer a heavier material. additionally, the lack of a pocket might be a minor drawback for some.overall, this polo shirt offers excellent value for its price, combining quality construction, stylish design, and versatile comfort. highly recommended for anyone looking for a dependable and affordable wardrobe staple.‚úö pros:- lightweight, breathable, and moisture-wicking fabric- slim but comfortable fit with well-constructed collar- wrinkle-resistant and easy to maintain- vibrant and accurate color options- excellent value for the price‚ùå cons:- thin material may not suit colder weather- no pocket, which may be a drawback for some",stylish and lightweight coofandy polo shirt,False,coofandy men polo shirt fantastic blend style comfort affordability making great choice casual smartcasual occasion made polyester shirt lightweight breathable moisturewicking ideal warmer weather active wear fit slim clingy shorter sleeve hug upper arm giving tailored modern look collar wellconstructed add polished touch without overbearingthe shirt material cool touch resists wri

In [119]:
# Display dataset information

print(
    f"Number of sellers : {amazon_df['seller_name'].nunique()}"
)

print(
    f"Number of products : {amazon_df['asin'].nunique()}"
)

display(
    amazon_df.describe(
        include="all"
    ).T
)

Number of sellers : 261
Number of products : 700


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
asin,6327,700,B074KL8RVS,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
about_item,6327,685,"comfort flex waistband: constructed with comfort in mind - our innovative flex waistband with stretch denim bands ensure a comfortable fit that moves and bends with youregular fit: built with a regular fit seat and thigh, these five-pocket regular fit jeans sit at the natural waist for a comfortable fitdurable materials: made with durable and comfortable flex denim for added ease of movement, these versatile jeans are made to last through everyday weareveryday five-pocket style: this everyday jean with a comfort waist takes you from the office and out to date night, keeping you feeling and looking greatheavy-duty hardware: finished with a zipper fly, button closure and our trademark embroidered back pockets",33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_description,2340,233,"we listen to customer feedback and fine-tune every detail to ensure our clothes are more comfortable, higher quality, and longer lasting—at affordable prices for the whole family.",73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
availability,6327,12,In Stock,5721,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand_name,6327,278,Hanes Store,318,NaN,NaN,NaN,NaN,NaN,NaN,NaN
manufacturer,6327,129,unknown,3968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_value,6327.0000,NaN,NaN,NaN,34.1253,22.9522,5.4573,19.9900,28.7928,40.0118,174.9900
rating_count,6327.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
rating_stars,6327.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
recent_purchases,6327.0000,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


# PHẦN 2. LOAD ESG KNOWLEDGE BASE

Nạp toàn bộ ESG Knowledge Base được xây dựng từ Notebook 03.

Các Knowledge Base đóng vai trò là nền tảng để chuyển đổi dữ liệu văn bản thành các đặc trưng ESG định lượng.

Bao gồm:

- ESG Keyword Dictionary
- Sustainable Material Dictionary
- Eco Label Dictionary
- Fashion ESG Complaint Ontology

In [120]:
# Load ESG Keyword Dictionary

esg_keyword_dictionary_df = pd.read_csv(
    esg_keyword_dictionary_path
)

print(
    f"Dataset shape: {esg_keyword_dictionary_df.shape}"
)

display(
    esg_keyword_dictionary_df.head()
)

Dataset shape: (45, 4)


,keyword,ESG_dimension,frequency,source
0,employee,Social,98384,ESG Sustainability Reports
1,risk,Governance,64645,ESG Sustainability Reports
2,energy,Environmental,54235,ESG Sustainability Reports
3,community,Social,53970,ESG Sustainability Reports
4,management,Governance,51008,ESG Sustainability Reports


In [121]:
# Load Sustainable Material Dictionary

material_dictionary_df = pd.read_csv(
    sustainable_material_dictionary_path
)

print(
    f"Dataset shape: {material_dictionary_df.shape}"
)

display(
    material_dictionary_df.head()
)

Dataset shape: (14, 3)


,material,frequency,source
0,polyester,148,Fast Fashion Material Composition Dataset
1,elastane,91,Fast Fashion Material Composition Dataset
2,viscose,86,Fast Fashion Material Composition Dataset
3,cotton,76,Fast Fashion Material Composition Dataset
4,polyamide,15,Fast Fashion Material Composition Dataset


In [122]:
# Load Eco Label Dictionary

eco_label_dictionary_df = pd.read_csv(
    eco_label_dictionary_path
)

print(
    f"Dataset shape: {eco_label_dictionary_df.shape}"
)

display(
    eco_label_dictionary_df.head()
)

Dataset shape: (9, 3)


,eco_label,frequency,source
0,organic cotton,50,Fast Fashion Eco Label Dataset
1,recycled polyester,17,Fast Fashion Eco Label Dataset
2,lyocell,8,Fast Fashion Eco Label Dataset
3,join life viscose,7,Fast Fashion Eco Label Dataset
4,water saving production,5,Fast Fashion Eco Label Dataset


In [123]:
# Load Fashion ESG Complaint Ontology

complaint_ontology_df = pd.read_csv(
    fashion_esg_complaint_ontology_path
)

print(
    f"Dataset shape: {complaint_ontology_df.shape}"
)

display(
    complaint_ontology_df.head()
)

Dataset shape: (7, 5)


,complaint_category,normalized_complaint,esg_dimension,frequency,source
0,Product Quality,poor quality,S,25746,Amazon Fashion Reviews Dataset
1,Product Sizing,wrong size,G,22047,Amazon Fashion Reviews Dataset
2,Product Damage,product damage,S,6287,Amazon Fashion Reviews Dataset
3,Counterfeit,counterfeit product,G,1367,Amazon Fashion Reviews Dataset
4,Customer Service,poor customer service,G,1094,Amazon Fashion Reviews Dataset


In [124]:
# ESG Knowledge Base Summary

knowledge_base_summary_df = pd.DataFrame(
    {
        "Knowledge Base": [
            "ESG Keyword Dictionary",
            "Sustainable Material Dictionary",
            "Eco Label Dictionary",
            "Fashion ESG Complaint Ontology"
        ],
        "Records": [
            len(esg_keyword_dictionary_df),
            len(material_dictionary_df),
            len(eco_label_dictionary_df),
            len(complaint_ontology_df)
        ]
    }
)

display(
    knowledge_base_summary_df
)

,Knowledge Base,Records
0,ESG Keyword Dictionary,45
1,Sustainable Material Dictionary,14
2,Eco Label Dictionary,9
3,Fashion ESG Complaint Ontology,7


# PHẦN 3. LOAD FEATURE SPECIFICATION

Nạp Feature Specification được xây dựng trong Notebook 04.

Feature Specification mô tả toàn bộ các đặc trưng ESG sẽ được xây dựng, bao gồm:

- ESG Dimension
- Data Type
- Measurement Method
- Knowledge Base sử dụng

Đây là tài liệu tham chiếu cho toàn bộ quá trình Feature Engineering.

In [125]:
# Load Feature Specification

feature_specification_df = pd.read_csv(
    feature_specification_path
)

print(
    f"Dataset shape: {feature_specification_df.shape}"
)

display(
    feature_specification_df
)

Dataset shape: (10, 5)


,Feature,ESG Dimension,Data Type,Measurement,Knowledge Base
0,environmental_keyword_count,E,Integer,Keyword Frequency,ESG Keyword Dictionary
1,sustainable_material_count,E,Integer,Dictionary Frequency,Sustainable Material Dictionary
2,eco_label_count,E,Integer,Dictionary Frequency,Eco Label Dictionary
3,product_quality_complaint_count,S,Integer,Complaint Frequency,Fashion ESG Complaint Ontology
4,product_damage_complaint_count,S,Integer,Complaint Frequency,Fashion ESG Complaint Ontology
5,product_safety_complaint_count,S,Integer,Complaint Frequency,Fashion ESG Complaint Ontology
6,customer_service_complaint_count,G,Integer,Complaint Frequency,Fashion ESG Complaint Ontology
7,counterfeit_complaint_count,G,Integer,Complaint Frequency,Fashion ESG Complaint Ontology
8,packaging_complaint_count,E,Integer,Complaint Frequency,Fashion ESG Complaint Ontology
9,governance_keyword_count,G,Integer,Keyword + Complaint Frequency,ESG Keyword Dictionary + Fashion ESG Complaint Ontology


In [126]:
# Feature Specification Summary

feature_summary_df = (
    feature_specification_df
    .groupby(
        "ESG Dimension"
    )
    .size()
    .reset_index(
        name="Number of Features"
    )
)

display(
    feature_summary_df
)

,ESG Dimension,Number of Features
0,E,4
1,G,3
2,S,3


# PHẦN 4. ENVIRONMENTAL FEATURE ENGINEERING

Xây dựng các đặc trưng thuộc nhóm Environmental (E) dựa trên thông tin mô tả sản phẩm.

Các đặc trưng được xây dựng bằng phương pháp Dictionary Matching giữa nội dung sản phẩm và ESG Knowledge Base.

Các Environmental Features bao gồm:

- environmental_keyword_count
- sustainable_material_count
- eco_label_count

Những đặc trưng này phản ánh mức độ xuất hiện của các yếu tố liên quan đến môi trường trong thông tin sản phẩm.

In [127]:
# Create product text

amazon_df["product_information"] = (
    amazon_df["about_item"]
    .fillna("")
    + " "
    + amazon_df["product_description"]
    .fillna("")
)

amazon_df["product_information"] = (
    amazon_df["product_information"]
    .str.lower()
)

In [128]:
# ESG environmental keywords

environmental_keywords = (
    esg_keyword_dictionary_df
    .loc[
        esg_keyword_dictionary_df["ESG_dimension"] == "E",
        "keyword"
    ]
    .str.lower()
    .tolist()
)

# Sustainable materials

material_keywords = (
    material_dictionary_df["material"]
    .str.lower()
    .tolist()
)

# Eco labels

eco_label_keywords = (
    eco_label_dictionary_df["eco_label"]
    .str.lower()
    .tolist()
)

In [129]:
# Count unique ESG indicators appearing in text

def keyword_count(
    text,
    keyword_list
):

    if pd.isna(text):

        return 0

    matched_keywords = []

    for keyword in keyword_list:

        if keyword in text:

            matched_keywords.append(
                keyword
            )

    return len(
        set(matched_keywords)
    )

In [130]:
# Environmental features

environment_feature_df = amazon_df.copy()

environment_feature_df["environmental_keyword_count"] = (
    environment_feature_df["product_information"]
    .apply(
        lambda x:
        keyword_count(
            x,
            environmental_keywords
        )
    )
)

environment_feature_df["sustainable_material_count"] = (
    environment_feature_df["product_information"]
    .apply(
        lambda x:
        keyword_count(
            x,
            material_keywords
        )
    )
)

environment_feature_df["eco_label_count"] = (
    environment_feature_df["product_information"]
    .apply(
        lambda x:
        keyword_count(
            x,
            eco_label_keywords
        )
    )
)

In [131]:
environment_feature_df[
    [
        "asin",
        "seller_name",
        "environmental_keyword_count",
        "sustainable_material_count",
        "eco_label_count"
    ]
].head()

,asin,seller_name,environmental_keyword_count,sustainable_material_count,eco_label_count
0,B0DLGB4RYH,COOFANDY,0,1,0
1,B0DRXF62JH,ZITY®,0,0,0
2,B0DRXF62JH,ZITY®,0,0,0
3,B0DRXF62JH,ZITY®,0,0,0
4,B0DRXF62JH,ZITY®,0,0,0


In [132]:
# Save environmental feature checkpoint

environment_feature_path = (
    project_path
    / "Dataset"
    / "processed"
    / "environment_feature_dataset.csv"
)


environment_feature_df.to_csv(
    environment_feature_path,
    index=False,
    encoding="utf-8-sig"
)

# PHẦN 5. SOCIAL & GOVERNANCE COMPLAINT FEATURE ENGINEERING

Xây dựng các đặc trưng ESG thuộc nhóm Social (S) và Governance (G) dựa trên nội dung đánh giá sản phẩm của khách hàng.

Các đặc trưng được tạo bằng phương pháp dictionary matching giữa:

- Amazon Fashion Reviews Dataset
- Fashion ESG Complaint Ontology

Do review của khách hàng thường sử dụng nhiều cách diễn đạt khác nhau cho cùng một vấn đề, quá trình matching sử dụng complaint keyword dictionary mở rộng thay vì chỉ sử dụng nhãn complaint chuẩn hóa.

Các Social Indicators:

- product_quality_complaint_count
- product_damage_complaint_count
- product_safety_complaint_count

Các Governance Indicators:

- customer_service_complaint_count
- counterfeit_complaint_count

In [133]:
# Prepare review text

social_governance_feature_df = amazon_df.copy()


social_governance_feature_df[
    "review_information"
] = (
    social_governance_feature_df[
        "cleaned_review_text"
    ]
    .fillna("")
    .str.lower()
)

In [134]:
# Complaint keyword dictionary

complaint_keywords = {


    # Social - Product Quality

    "product_quality_complaint_count": [

        "poor quality",
        "terrible quality",
        "bad quality",
        "cheap quality",
        "low quality",
        "cheap material",
        "poor stitching",
        "terrible material",
        "thin material",
        "low grade material"

    ],



    # Social - Product Damage

    "product_damage_complaint_count": [

        "fell apart",
        "fall apart",
        "came apart",
        "ripped apart",
        "broken zipper",
        "broken button",
        "broken strap",
        "torn seam",
        "loose thread",
        "loose stitching"

    ],



    # Social - Product Safety

    "product_safety_complaint_count": [

        "chemical smell",
        "strange smell",
        "bad smell",
        "weird smell",
        "odor",
        "smells bad"

    ],



    # Governance - Customer Service

    "customer_service_complaint_count": [

        "customer service",
        "poor service",
        "bad service",
        "seller support",
        "no response"

    ],



    # Governance - Counterfeit

    "counterfeit_complaint_count": [

        "counterfeit",
        "fake product",
        "fake item",
        "not authentic",
        "buyer beware"

    ]

}

In [135]:
# Complaint indicator function

def complaint_indicator(
    text,
    keywords
):

    if pd.isna(text):

        return 0


    for keyword in keywords:

        if keyword in text:

            return 1


    return 0

In [136]:
# Generate Social and Governance features

for feature_name, keyword_list in complaint_keywords.items():

    social_governance_feature_df[
        feature_name
    ] = (
        social_governance_feature_df[
            "review_information"
        ]
        .apply(
            lambda x:
            complaint_indicator(
                x,
                keyword_list
            )
        )
    )

In [137]:
# Display generated features

display(
    social_governance_feature_df[
        [
            "asin",
            "seller_name",
            "product_quality_complaint_count",
            "product_damage_complaint_count",
            "product_safety_complaint_count",
            "customer_service_complaint_count",
            "counterfeit_complaint_count"
        ]
    ]
    .head()
)

,asin,seller_name,product_quality_complaint_count,product_damage_complaint_count,product_safety_complaint_count,customer_service_complaint_count,counterfeit_complaint_count
0,B0DLGB4RYH,COOFANDY,1,0,0,0,0
1,B0DRXF62JH,ZITY®,0,0,0,0,0
2,B0DRXF62JH,ZITY®,0,0,0,0,0
3,B0DRXF62JH,ZITY®,0,1,0,0,0
4,B0DRXF62JH,ZITY®,0,0,0,0,0


In [138]:
# Complaint summary

complaint_summary_df = (
    social_governance_feature_df[
        list(
            complaint_keywords.keys()
        )
    ]
    .sum()
    .reset_index()
)


complaint_summary_df.columns = [
    "Feature",
    "Total Complaints"
]


display(
    complaint_summary_df
)

,Feature,Total Complaints
0,product_quality_complaint_count,58
1,product_damage_complaint_count,39
2,product_safety_complaint_count,18
3,customer_service_complaint_count,10
4,counterfeit_complaint_count,6


In [139]:
# Save environmental feature checkpoint

social_governance_feature_path = (
    project_path
    / "Dataset"
    / "processed"
    / "social_governance_feature_dataset.csv"
)


social_governance_feature_df.to_csv(
    social_governance_feature_path,
    index=False,
    encoding="utf-8-sig"
)

# PHẦN 6. GOVERNANCE KEYWORD FEATURE ENGINEERING

Xây dựng Governance indicator dựa trên sự xuất hiện của các từ khóa liên quan đến quản trị doanh nghiệp trong thông tin sản phẩm và đánh giá khách hàng.

Feature được tạo:

- governance_keyword_count

Feature này phản ánh mức độ xuất hiện của các yếu tố liên quan đến:

- Product authenticity
- Transparency
- Responsible business practice
- Customer commitment

Nguồn dữ liệu:

- ESG Keyword Dictionary
- Amazon Product Information
- Amazon Review Text

In [140]:
# Check ESG keyword dictionary

display(
    esg_keyword_dictionary_df.head()
)

,keyword,ESG_dimension,frequency,source
0,employee,Social,98384,ESG Sustainability Reports
1,risk,Governance,64645,ESG Sustainability Reports
2,energy,Environmental,54235,ESG Sustainability Reports
3,community,Social,53970,ESG Sustainability Reports
4,management,Governance,51008,ESG Sustainability Reports


In [141]:
# Prepare governance keywords

governance_keywords = (

    esg_keyword_dictionary_df[
        "keyword"
    ]
    .str.lower()
    .tolist()

)

In [142]:
# Combine product and review information

social_governance_feature_df[
    "governance_information"
] = (

    social_governance_feature_df[
        "product_information"
    ]
    .fillna("")

    + " "

    +

    social_governance_feature_df[
        "review_information"
    ]
    .fillna("")

)

In [143]:
# Governance keyword indicator

def governance_keyword_indicator(
    text,
    keyword_list
):

    if pd.isna(text):

        return 0


    matched_keyword = False


    for keyword in keyword_list:

        if keyword in text:

            matched_keyword = True

            break


    return int(
        matched_keyword
    )

In [144]:
# Generate governance keyword feature

social_governance_feature_df[
    "governance_keyword_count"
] = (

    social_governance_feature_df[
        "governance_information"
    ]
    .apply(
        lambda x:
        governance_keyword_indicator(
            x,
            governance_keywords
        )
    )

)

In [145]:
display(

    social_governance_feature_df[
        [
            "asin",
            "seller_name",
            "governance_keyword_count"
        ]
    ]
    .head()

)

,asin,seller_name,governance_keyword_count
0,B0DLGB4RYH,COOFANDY,1
1,B0DRXF62JH,ZITY®,0
2,B0DRXF62JH,ZITY®,0
3,B0DRXF62JH,ZITY®,0
4,B0DRXF62JH,ZITY®,1


In [146]:
governance_summary_df = (

    social_governance_feature_df[
        [
            "governance_keyword_count"
        ]
    ]
    .sum()
    .reset_index()

)


governance_summary_df.columns = [

    "Feature",
    "Total Match"

]


display(
    governance_summary_df
)

,Feature,Total Match
0,governance_keyword_count,3788


In [147]:
# Save environmental feature checkpoint

governance_feature_path = (
    project_path
    / "Dataset"
    / "processed"
    / "social_governance_feature_dataset.csv"
)

social_governance_feature_df.to_csv(
    governance_feature_path,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    governance_feature_path
)

Saved: ..\Dataset\processed\social_governance_feature_dataset.csv


# PHẦN 7. ESG FEATURE CONSOLIDATION & SELLER-LEVEL AGGREGATION

Tổng hợp toàn bộ ESG operational features từ cấp sản phẩm/review về cấp nhà bán hàng (seller level).

Do mục tiêu nghiên cứu là xây dựng mô hình chấm điểm ESG cho nhà bán hàng thời trang trên sàn thương mại điện tử, đơn vị đánh giá cuối cùng là seller.

Dataset đầu ra được sử dụng cho:

- AHP Weighting
- Seller ESG Score Calculation
- Machine Learning Modeling

In [148]:
# Load environmental features

environment_feature_df = pd.read_csv(
    environment_feature_path
)


# Load social governance features

social_governance_feature_df = pd.read_csv(
    social_governance_feature_path
)

In [149]:
# Check environmental features

display(
    environment_feature_df.head()
)

,asin,about_item,product_description,availability,brand_name,manufacturer,price_value,rating_count,rating_stars,recent_purchases,seller_name,seller_page_url,rank_1,best_sellers_rank,productasin,productvariant,rating,reviewid,reviewmetadata,reviewtext,reviewtitle,verifiedpurchase,cleaned_review_text,sentiment_score,product_text,review_text_clean,has_review,has_product_info,review_length,product_text_length,is_verified_purchase,product_information,environmental_keyword_count,sustainable_material_count,eco_label_count
0,B0DLGB4RYH,"material: men's polo shirt is made of soft polyester fabric, moisture wicking, stain-resistant, durable work shirts, lightweight, breathable and comfortable. golf shirt for men keeps you cool and dry in all day.design: mens short sleeve polo shirts designed with 3 button closure, solid color basic collared t-shirt with split hem, unrestricted movement during your golf swings, fashion classic design golf polo shirt is great for spring, summer and fall.match: polo shirts can be worn out or tucked. tucked polos for business casual settings, while untucked polos casual look ideal for everyday wear. basic polo t shirts for men easy to match with slacks, jeans, suit pants, shorts, etc.occasions: polo shirts for men is suitable for casual, golf, business, office, work, everyday wear, school uniform, travel, meeting, dating, dinner, street, hiking, camping, tennis or other activities. perfect valentine's day, father's day, christmas, thanksgiving day, birthday gift for father, husband, son, boyfriends, friends or yourself.garment care: machine washable. easy to care and wash. note: men's golf polo shirt is in standard us size, please choose your size refer to the size information under the description before ordering.",NaN,In Stock,COOFANDY Store,unknown,19.9920,0.0000,0.0000,0.0000,COOFANDY,https://www.amazon.com/gp/help/seller/at-a-glance.html/ref=dp_merchant_link?ie=UTF8&seller=AW10J8VB8Z74G&asin=B0DLGB4RYH&ref_=dp_merchant_link&isAmazonFulfilled=1,199.0000,"#50,261 in Clothing, Shoes & Jewelry (See Top 100 in Clothing, Shoes & Jewelry) #199 in Men's Polo Shirts",B0DLGB4RYH,Color: BlackSize: X-Large,5.0000,R2AUQFPJY5ERCZ,"Reviewed in the United States on March 6, 2025","‚úçô∏è the coofandy men's polo shirt is a fantastic blend of style, comfort, and affordability, making it a great choice for casual or smart-casual occasions. made from 100% polyester, the shirt is lightweight, breathable, and moisture-wicking, ideal for warmer weather or active wear. the fit is slim but not clingy, with shorter sleeves that hug the upper arms, giving it a tailored and modern look. the collar is well-constructed and adds a polished touch without being overbearing.the shirt‚äôs material is cool to the touch and resists wrinkling, making it easy to maintain. the color options, like light blue, dark gray, and wine red, are vibrant and true to the product photos. however, the fabric is on the thinner side, so it may not be suitable for colder weather or those who prefer a heavier material. additionally, the lack of a pocket might be a minor drawback for some.overall, this polo shirt offers excellent value for its price, combining quality construction, stylish design, and versatile comfort. highly recommended for anyone looking for a dependable and affordable wardrobe staple.‚úö pros:- lightweight, breathable, and moisture-wicking fabric- slim but comfortable fit with well-constructed collar- wrinkle-resistant and easy to maintain- vibrant and accurate color options- excellent value for the price‚ùå cons:- thin material may not suit colder weather- no pocket, which may be a drawback for some",stylish and lightweight coofandy polo shirt,False,coofandy men polo shirt fantastic blend style comfort affordability making great choice casual smartcasual occasion made polyester shirt lightweight breathable moisturewicking ideal warmer weather active wear fit slim clingy shorter sleeve hug upper arm giving tailored modern look collar well

In [150]:
# Check social governance features

display(
    social_governance_feature_df.head()
)

,asin,about_item,product_description,availability,brand_name,manufacturer,price_value,rating_count,rating_stars,recent_purchases,seller_name,seller_page_url,rank_1,best_sellers_rank,productasin,productvariant,rating,reviewid,reviewmetadata,reviewtext,reviewtitle,verifiedpurchase,cleaned_review_text,sentiment_score,product_text,review_text_clean,has_review,has_product_info,review_length,product_text_length,is_verified_purchase,product_information,review_information,product_quality_complaint_count,product_damage_complaint_count,product_safety_complaint_count,customer_service_complaint_count,counterfeit_complaint_count,governance_information,governance_keyword_count
0,B0DLGB4RYH,"material: men's polo shirt is made of soft polyester fabric, moisture wicking, stain-resistant, durable work shirts, lightweight, breathable and comfortable. golf shirt for men keeps you cool and dry in all day.design: mens short sleeve polo shirts designed with 3 button closure, solid color basic collared t-shirt with split hem, unrestricted movement during your golf swings, fashion classic design golf polo shirt is great for spring, summer and fall.match: polo shirts can be worn out or tucked. tucked polos for business casual settings, while untucked polos casual look ideal for everyday wear. basic polo t shirts for men easy to match with slacks, jeans, suit pants, shorts, etc.occasions: polo shirts for men is suitable for casual, golf, business, office, work, everyday wear, school uniform, travel, meeting, dating, dinner, street, hiking, camping, tennis or other activities. perfect valentine's day, father's day, christmas, thanksgiving day, birthday gift for father, husband, son, boyfriends, friends or yourself.garment care: machine washable. easy to care and wash. note: men's golf polo shirt is in standard us size, please choose your size refer to the size information under the description before ordering.",NaN,In Stock,COOFANDY Store,unknown,19.9920,0.0000,0.0000,0.0000,COOFANDY,https://www.amazon.com/gp/help/seller/at-a-glance.html/ref=dp_merchant_link?ie=UTF8&seller=AW10J8VB8Z74G&asin=B0DLGB4RYH&ref_=dp_merchant_link&isAmazonFulfilled=1,199.0000,"#50,261 in Clothing, Shoes & Jewelry (See Top 100 in Clothing, Shoes & Jewelry) #199 in Men's Polo Shirts",B0DLGB4RYH,Color: BlackSize: X-Large,5.0000,R2AUQFPJY5ERCZ,"Reviewed in the United States on March 6, 2025","‚úçô∏è the coofandy men's polo shirt is a fantastic blend of style, comfort, and affordability, making it a great choice for casual or smart-casual occasions. made from 100% polyester, the shirt is lightweight, breathable, and moisture-wicking, ideal for warmer weather or active wear. the fit is slim but not clingy, with shorter sleeves that hug the upper arms, giving it a tailored and modern look. the collar is well-constructed and adds a polished touch without being overbearing.the shirt‚äôs material is cool to the touch and resists wrinkling, making it easy to maintain. the color options, like light blue, dark gray, and wine red, are vibrant and true to the product photos. however, the fabric is on the thinner side, so it may not be suitable for colder weather or those who prefer a heavier material. additionally, the lack of a pocket might be a minor drawback for some.overall, this polo shirt offers excellent value for its price, combining quality construction, stylish design, and versatile comfort. highly recommended for anyone looking for a dependable and affordable wardrobe staple.‚úö pros:- lightweight, breathable, and moisture-wicking fabric- slim but comfortable fit with well-constructed collar- wrinkle-resistant and easy to maintain- vibrant and accurate color options- excellent value for the price‚ùå cons:- thin material may not suit colder weather- no pocket, which may be a drawback for some",stylish and lightweight coofandy polo shirt,False,coofandy men polo shirt fantastic blend style comfort affordability making great choice casual smartcasual occasion made polyester shirt light

In [151]:
# Environmental feature columns

environment_columns = [

    "asin",
    "seller_name",

    "environmental_keyword_count",
    "sustainable_material_count",
    "eco_label_count"

]


environment_feature_df = (
    environment_feature_df[
        environment_columns
    ]
)

In [152]:
# Social governance feature columns

social_governance_columns = [

    "asin",
    "seller_name",

    "product_quality_complaint_count",
    "product_damage_complaint_count",
    "product_safety_complaint_count",

    "customer_service_complaint_count",
    "counterfeit_complaint_count",

    "governance_keyword_count"

]


social_governance_feature_df = (
    social_governance_feature_df[
        social_governance_columns
    ]
)

In [153]:
# Merge ESG features

esg_feature_df = pd.merge(

    environment_feature_df,

    social_governance_feature_df,

    on=[
        "asin",
        "seller_name"
    ],

    how="outer"

)


display(
    esg_feature_df.head()
)

,asin,seller_name,environmental_keyword_count,sustainable_material_count,eco_label_count,product_quality_complaint_count,product_damage_complaint_count,product_safety_complaint_count,customer_service_complaint_count,counterfeit_complaint_count,governance_keyword_count
0,B00021NY28,Amazon.com,0,0,0,0,0,0,0,0,1
1,B00021NY28,Amazon.com,0,0,0,0,0,0,0,0,0
2,B00021NY28,Amazon.com,0,0,0,0,0,0,0,0,1
3,B00021NY28,Amazon.com,0,0,0,0,0,0,0,0,0
4,B00021NY28,Amazon.com,0,0,0,0,0,0,0,0,0


In [154]:
# Fill missing ESG indicators

esg_feature_columns = [

    "environmental_keyword_count",
    "sustainable_material_count",
    "eco_label_count",

    "product_quality_complaint_count",
    "product_damage_complaint_count",
    "product_safety_complaint_count",

    "customer_service_complaint_count",
    "counterfeit_complaint_count",

    "governance_keyword_count"

]


esg_feature_df[
    esg_feature_columns
] = (

    esg_feature_df[
        esg_feature_columns
    ]
    .fillna(0)

)

In [155]:
# Aggregate ESG features by seller

seller_esg_feature_df = (

    esg_feature_df

    .groupby(
        "seller_name"
    )

    [
        esg_feature_columns
    ]

    .mean()

    .reset_index()

)


display(
    seller_esg_feature_df.head()
)

,seller_name,environmental_keyword_count,sustainable_material_count,eco_label_count,product_quality_complaint_count,product_damage_complaint_count,product_safety_complaint_count,customer_service_complaint_count,counterfeit_complaint_count,governance_keyword_count
0,101Dealz,0.0000,1.6667,0.0000,0.0000,0.0333,0.0667,0.0000,0.0000,0.5000
1,123(EXOPRT WAREHOUSE),0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
2,24 Caret,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.1000
3,"33,000ft outdoor",0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000
4,5Mayi Official,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000


In [156]:
# Seller count

print(
    "Number of sellers:",
    seller_esg_feature_df[
        "seller_name"
    ]
    .nunique()
)

Number of sellers: 261


In [160]:
# Save environmental feature checkpoint

seller_esg_feature_path = (
    project_path
    / "Dataset"
    / "processed"
    / "seller_esg_feature_dataset.csv"
)


seller_esg_feature_df.to_csv(
    seller_esg_feature_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    seller_esg_feature_path
)

Saved: ..\Dataset\processed\seller_esg_feature_dataset.csv
